<a href="https://colab.research.google.com/github/S-Chakraborty163/Learning-DL/blob/main/optuna_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 7.3 MB/s eta 0:00:00


In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [23]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-01-18 13:32:07,500] A new study created in memory with name: no-name-dc1e76ea-e259-4971-b069-9d0c8333ef7f
[I 2026-01-18 13:32:07,994] Trial 0 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 93, 'max_depth': 6}. Best is trial 0 with value: 0.7597765363128491.
[I 2026-01-18 13:32:08,611] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 108, 'max_depth': 15}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-01-18 13:32:09,590] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 177, 'max_depth': 20}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-01-18 13:32:10,298] Trial 3 finished with value: 0.7783985102420856 and parameters: {'n_estimators': 131, 'max_depth': 15}. Best is trial 3 with value: 0.7783985102420856.
[I 2026-01-18 13:32:10,694] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 74, 'max_depth': 5}. Best is trial 3 with value: 0.77839851

In [24]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7839851024208566
Best hyperparameters: {'n_estimators': 73, 'max_depth': 18}


In [25]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.76


## Samplers in Optuna

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [10]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-01-18 13:27:55,638] A new study created in memory with name: no-name-bfa1f47d-4153-42a0-b054-939922f0c9a8
[I 2026-01-18 13:27:55,939] Trial 0 finished with value: 0.7597765363128492 and parameters: {'n_estimators': 53, 'max_depth': 12}. Best is trial 0 with value: 0.7597765363128492.
[I 2026-01-18 13:27:57,154] Trial 1 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 182, 'max_depth': 5}. Best is trial 1 with value: 0.7635009310986964.
[I 2026-01-18 13:27:58,112] Trial 2 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 111, 'max_depth': 18}. Best is trial 2 with value: 0.7765363128491621.
[I 2026-01-18 13:27:59,279] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 166, 'max_depth': 11}. Best is trial 2 with value: 0.7765363128491621.
[I 2026-01-18 13:27:59,815] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 103, 'max_depth': 5}. Best is trial 2 with value: 0.7765363

In [11]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 107, 'max_depth': 12}


In [12]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


In [13]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [14]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-01-18 13:29:06,057] A new study created in memory with name: no-name-93c3b29f-a267-4d44-8b0d-ac87e951e224
[I 2026-01-18 13:29:06,588] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-01-18 13:29:07,389] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-01-18 13:29:07,696] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-01-18 13:29:08,246] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-01-18 13:29:08,974] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [15]:

print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [16]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations(On TPE Sampler case)

In [26]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [27]:
# 1. Optimization History
plot_optimization_history(study).show()

In [28]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [29]:
# 3. Slice Plot
plot_slice(study).show()

In [30]:
# 4. Contour Plot
plot_contour(study).show()

In [31]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [32]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [33]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [34]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-01-18 13:40:09,159] A new study created in memory with name: no-name-8d020ccc-a8c0-4bb3-8403-352edfe5334e
[I 2026-01-18 13:40:12,069] Trial 0 finished with value: 0.7728119180633147 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 216, 'learning_rate': 0.01150874931646131, 'max_depth': 13, 'min_samples_split': 3, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-01-18 13:40:13,132] Trial 1 finished with value: 0.7616387337057727 and parameters: {'classifier': 'RandomForest', 'n_estimators': 130, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 7, 'bootstrap': True}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-01-18 13:40:13,190] Trial 2 finished with value: 0.7653631284916201 and parameters: {'classifier': 'SVM', 'C': 0.10915854557383498, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-01-18 13:40:15,900] Trial 3 finished with value: 0.7318435754189944 and para

In [35]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11323498569454386, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [36]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.772812,2026-01-18 13:40:09.162079,2026-01-18 13:40:12.069746,0 days 00:00:02.907667,NaN,NaN,GradientBoosting,NaN,NaN,0.011509,13.0,7.0,3.0,216.0,COMPLETE
1,1,0.761639,2026-01-18 13:40:12.070935,2026-01-18 13:40:13.132483,0 days 00:00:01.061548,NaN,True,RandomForest,NaN,NaN,NaN,9.0,7.0,5.0,130.0,COMPLETE
2,2,0.765363,2026-01-18 13:40:13.134340,2026-01-18 13:40:13.190614,0 days 00:00:00.056274,0.109159,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.731844,2026-01-18 13:40:13.191707,2026-01-18 13:40:15.900469,0 days 00:00:02.708762,NaN,NaN,GradientBoosting,NaN,NaN,0.043232,10.0,2.0,5.0,118.0,COMPLETE
4,4,0.759777,2026-01-18 13:40:15.901353,2026-01-18 13:40:16.742538,0 days 00:00:00.841185,NaN,True,RandomForest,NaN,NaN,NaN,6.0,10.0,10.0,170.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.772812,2026-01-18 13:40:53.156335,2026-01-18 13:40:53.200017,0 days 00:00:00.043682,0.170083,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.728119,2026-01-18 13:40:53.200861,2026-01-18 13:40:53.239536,0 days 00:00:00.038675,0.314725,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.789572,2026-01-18 13:40:53.240323,2026-01-18 13:40:53.269170,0 days 00:00:00.028847,0.125871,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.787709,2026-01-18 13:40:53.270001,2026-01-18 13:40:53.301025,0 days 00:00:00.031024,0.163633,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [37]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,77
GradientBoosting,12
RandomForest,11


In [38]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.748759
RandomForest,0.763840
SVM,0.774989


In [39]:
# 1. Optimization History
plot_optimization_history(study).show()

In [40]:
# 3. Slice Plot
plot_slice(study).show()

In [41]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()